# **Notebook 03 — Statistical Analysis**

**Purpose.** This notebook serves as the **statistical referee** for Notebook 02. It does not introduce a new set of descriptive charts; instead, it evaluates the claims already identified in the descriptive analysis. For each major finding, the notebook determines whether the evidence is statistically credible, quantifies the magnitude and uncertainty of the relationship, tests whether it remains after appropriate controls, and defines what can—and cannot—be concluded from observational data.

## **Pre-registered hypotheses**

Four primary hypotheses are fixed **before testing**. All other analyses are exploratory and are explicitly labelled as such, reducing the risk that reported significance is driven by searching across many possible tests.

| ID | Hypothesis | Primary outcome |
|---|---|---|
| **H1** | Delivery lateness is associated with customer dissatisfaction | `review_score`, `poor_review` |
| **H2** | Seller–customer distance is associated with fulfilment burden | Delivery days, freight, review score |
| **H3** | First-order satisfaction predicts repeat purchase | 90-day repeat |
| **H4** | Cross-state fulfilment carries greater burden than within-state fulfilment | Distance, delivery time, freight |

Seller concentration is treated as a **descriptive concentration measure**, rather than a hypothesis test. There is no meaningful null hypothesis to reject about how concentrated a revenue distribution is; concentration is therefore quantified using measures such as seller-share ratios, Gini, and HHI.

## **Standing analytical rules**

**Grain.** All order-level outcomes use `v_fact_orders`, with **one row per order**. The earlier item-grain χ² result is discarded: at item grain, a six-item order contributes the same review score six times, artificially inflating the effective sample size and producing standard errors and p-values that are too small.

**Significance ≠ magnitude.** With approximately 96,000 customers/orders in the relevant analyses, even very small differences can produce very small p-values. Every primary test therefore reports an **effect size** and **95% confidence interval** alongside the p-value. Conclusions distinguish **statistical significance** from **practical significance**.

**Causal language.** Olist is observational and contains no randomized treatment assignment. Results are therefore described as **associated with**, **predicts**, or **differs across** rather than as causal effects. Causal conclusions are not drawn from these analyses.

**Multiple testing.** The four hypotheses above are the pre-specified confirmatory analyses. Pairwise post-hoc comparisons use **Holm correction**. Additional analyses are explicitly marked as **exploratory** and are not treated as confirmatory evidence.

**Interpretation standard.** A statistically significant result is not automatically considered important. Each finding must be evaluated in terms of its **direction, effect size, uncertainty, robustness, and business relevance**.

## 1 · Setup

In [3]:
# Install required packages
!pip install statsmodels scikit-posthocs --quiet

# Authenticate Google Colab
from google.colab import auth
auth.authenticate_user()

# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

import scikit_posthocs as sp
from scipy import stats

from google.cloud import bigquery

# BigQuery Configuration

PROJECT = "Project_ID"
DATASET = "olist"

client = bigquery.Client(project=PROJECT)

# BigQuery Query Helper

def q(sql):
    return client.query(
        sql.replace(
            "`olist.",
            f"`{PROJECT}.{DATASET}."
        )
    ).to_dataframe()

# Pandas Display Settings

pd.set_option(
    "display.float_format",
    lambda v: f"{v:,.4f}"
)

print("ready")

ready


# **2. Data Validation**

**No statistical test is run until the analytical grain is validated. This section is the gatekeeper for all downstream inference.**

In [4]:
df = q("""
    SELECT
        order_id,
        customer_unique_id,
        order_status,
        purchase_date,
        delivery_days,
        lateness_days,
        days_late_pos,
        late_flag,
        lateness_band,
        review_score,
        poor_review,
        item_value,
        freight_value,
        order_value,
        n_items,
        category,
        customer_state,
        dominant_seller_state,
        is_interstate,
        max_distance_km,
        order_seq,
        is_first_order,
        exposure_days,
        days_to_next_order,
        repeat_within_30d,
        repeat_within_60d,
        repeat_within_90d,
        repeat_within_180d,
        ever_repeated
    FROM `olist.v_fact_orders`
""")

print(f"rows                : {len(df):,}")
print(f"distinct order_id   : {df.order_id.nunique():,}")

assert len(df) == df.order_id.nunique(), \
    "GRAIN FAILURE: duplicate order_id"

print("grain OK - one row per order")

rows                : 98,666
distinct order_id   : 98,666
grain OK - one row per order


**Expected:** 98,666 rows, all distinct. If the assertion fails, something upstream isfanning out — most likely the geolocation join, which has ~52.6 rows per zip prefix andmust go through `v_geo_dedup`.

In [5]:
# ============================================================
# Analysis frame for H1/H2:
# Delivered orders with a review
# ============================================================

a = df[
    (df.order_status == "delivered")
    & df.review_score.notna()
    & df.lateness_days.notna()
].copy()

# ------------------------------------------------------------
# 1. Define lateness band order
# ------------------------------------------------------------

band_order = [
    "Early",
    "On time",
    "1-3d late",
    "4-7d late",
    "8d+ late"
]

a["lateness_band"] = pd.Categorical(
    a.lateness_band,
    categories=band_order,
    ordered=True
)

# ------------------------------------------------------------
# 2. Create log-transformed variables
# ------------------------------------------------------------

a["log_dist"] = np.log1p(a.max_distance_km)
a["log_price"] = np.log1p(a.item_value)
a["log_freight"] = np.log1p(a.freight_value)

# ------------------------------------------------------------
# 3. Keep top 12 categories
# ------------------------------------------------------------

top = a.category.value_counts().head(12).index

a["cat"] = np.where(
    a.category.isin(top),
    a.category,
    "other"
)

# ------------------------------------------------------------
# 4. Basic analysis summary
# ------------------------------------------------------------

print(f"analysis N = {len(a):,}")

# ------------------------------------------------------------
# 5. Missingness on modelling columns
# ------------------------------------------------------------

print("\nmissingness on modelling columns:")

print(
    a[
        [
            "review_score",
            "lateness_days",
            "delivery_days",
            "max_distance_km",
            "item_value",
            "freight_value"
        ]
    ].isna().sum()
)

# ------------------------------------------------------------
# 6. Lateness band distribution
# ------------------------------------------------------------

print("\nband distribution:")

print(
    a.lateness_band
    .value_counts()
    .reindex(band_order)
    .to_frame("orders")
    .assign(
        pct=lambda t: (
            100 * t.orders / t.orders.sum()
        ).round(2)
    )
)

analysis N = 95,824

missingness on modelling columns:
review_score         0
lateness_days        0
delivery_days        0
max_distance_km    475
item_value           0
freight_value        0
dtype: int64

band distribution:
               orders     pct
lateness_band                
Early           88163 92.0100
On time          1280  1.3400
1-3d late        1852  1.9300
4-7d late        1748  1.8200
8d+ late         2781  2.9000


**Expected:** **N ≈ 95,800 eligible orders.**

**Critical structural fact:** Approximately **92% of orders fall into the "Early" band**, reflecting the heavily padded delivery estimates in the Olist data. Most orders therefore arrive comfortably before the promised date.

This has three important implications for the statistical analysis:

1. **Highly unequal group sizes.** The Early group dominates the sample, so every comparison reports the underlying **n** alongside the estimated effect.
2. **Test selection matters.** Methods that rely on balanced groups should not be assumed appropriate; statistical procedures are chosen based on the outcome distribution and group structure rather than convention.
3. **"Late" represents an SLA breach, not simply slow delivery.** Because the promised dates are generally generous, orders classified as late have exceeded an already-buffered delivery window and may represent **failed fulfilment events** rather than ordinary variation in delivery speed.

This distinction is important when interpreting the relationship between lateness and customer satisfaction throughout the analysis.

---

# **3. H1 — Delivery Lateness and Dissatisfaction**

**Business question.** Is delivery reliability associated with customer dissatisfaction strongly enough to justify operational attention, and how large is the observed relationship?

**H₀.** The distribution of `review_score` is the same across all delivery-lateness bands.

**H₁.** At least one lateness band has a different `review_score` distribution.

**Outcomes.**
- `review_score` — ordinal score from 1 to 5.
- `poor_review` — binary indicator equal to **1 when `review_score ≤ 2`**, and 0 otherwise.

The analysis first tests whether review outcomes differ across lateness bands, then quantifies the **magnitude and uncertainty** of those differences before assessing whether the relationship remains after appropriate controls.

## **3.1 Assumption Check — Why Not ANOVA?**

`review_score` is **ordinal, bounded, and strongly bimodal**, with approximately **58% of observations rated 5** and **12% rated 1**. This structure is poorly suited to the assumptions typically invoked for ANOVA: a continuous, approximately normally distributed outcome with comparable variance across groups.

Rather than applying ANOVA by convention, the analysis uses **Kruskal–Wallis as the primary group-comparison test**. It is a non-parametric method suitable for comparing more than two independent groups without requiring a normal distribution of the outcome.

The principle is straightforward: **choose the statistical method to match the structure of the data, rather than the other way around.**

In [6]:
# Descriptive statistics by lateness band

groups = [
    a.review_score[
        a.lateness_band == b
    ].values
    for b in band_order
]

for b, g in zip(band_order, groups):
    print(
        f"{b:<12} "
        f"n={len(g):>7,}  "
        f"mean={g.mean():.3f}  "
        f"median={np.median(g):.0f}  "
        f"%poor={(g <= 2).mean() * 100:.1f}"
    )



# Normality and Equal Variance Tests

sub = a.review_score.sample(
    4000,
    random_state=0
)

print(
    "\nShapiro-Wilk on a 4,000-row sample:",
    f"p = {stats.shapiro(sub).pvalue:.3g}"
)

print(
    "Levene (equal variances):",
    f"p = {stats.levene(*groups).pvalue:.3g}"
)

Early        n= 88,163  mean=4.294  median=5  %poor=9.2
On time      n=  1,280  mean=4.034  median=5  %poor=12.4
1-3d late    n=  1,852  mean=3.291  median=4  %poor=32.1
4-7d late    n=  1,748  mean=2.105  median=1  %poor=67.6
8d+ late     n=  2,781  mean=1.696  median=1  %poor=79.3

Shapiro-Wilk on a 4,000-row sample: p = 5.24e-66
Levene (equal variances): p = 5.41e-181


Both assumption checks reject decisively, consistent with the strongly bimodal, ordinal structure of `review_score`. These results are documented to make the test selection explicit rather than relying on convention.

---

## **3.2 Primary Test: Kruskal–Wallis**

The **Kruskal–Wallis test** is used to determine whether the distribution of review scores differs across the five delivery-lateness bands.

In [7]:

# Kruskal-Wallis Test + Epsilon-Squared Effect Size

H, p = stats.kruskal(*groups)

k = len(groups)
n = len(a)

# Epsilon-squared effect size
eps2 = (H - k + 1) / (n - k)


print(
    f"Kruskal-Wallis  H = {H:,.1f}   "
    f"df = {k - 1}   "
    f"p = {p:.3g}"
)

print(
    f"epsilon^2 = {eps2:.4f}"
)

print(
    "\nBenchmarks: "
    "0.01 small | 0.06 medium | 0.14 large"
)

Kruskal-Wallis  H = 10,167.7   df = 4   p = 0
epsilon^2 = 0.1061

Benchmarks: 0.01 small | 0.06 medium | 0.14 large


## **Expected result:** **H ≈ 10,171, p < 10⁻³⁰⁰, ε² ≈ 0.106.**

## **Interpretation**

The extremely small p-value establishes that review-score distributions differ across delivery-lateness bands, but with approximately 96,000 observations, statistical significance alone is not informative about practical importance.

The key quantity is **ε² ≈ 0.106**, indicating a **substantial effect**: delivery-lateness bands account for roughly **10.6% of the variation in review-score ranks** in this analysis.

This is the more meaningful evidence for the project: **delivery performance is not merely statistically different across groups; the magnitude of the observed association is substantial for a single operational variable and an inherently subjective customer-satisfaction measure.**

The result should still be interpreted as an **association, not a causal effect**, because the Olist data are observational.

---

## **3.3 Post-hoc: Which Bands Actually Differ?**

The Kruskal–Wallis test establishes that **at least one delivery-lateness group differs**, but it does not identify where those differences occur.

To locate the specific differences, we use **Dunn's post-hoc test** for all 10 pairwise comparisons between the five lateness bands. Because multiple comparisons increase the chance of false positives, p-values are adjusted using the **Holm correction** to control the family-wise error rate.

The goal is to determine **which delivery-performance transitions are associated with meaningful changes in review outcomes**, rather than treating the five bands as a single undifferentiated effect.

In [8]:

# Dunn's Post-Hoc Test with Holm Adjustment

dunn = sp.posthoc_dunn(
    a,
    val_col="review_score",
    group_col="lateness_band",
    p_adjust="holm"
)

dunn = dunn.reindex(
    index=band_order,
    columns=band_order
)

print(
    "Dunn post-hoc, Holm-adjusted p-values:\n"
)

print(
    dunn.applymap(
        lambda x: f"{x:.3e}"
    ).to_string()
)

Dunn post-hoc, Holm-adjusted p-values:

                Early     On time   1-3d late   4-7d late    8d+ late
Early       1.000e+00   9.158e-18  1.100e-200   0.000e+00   0.000e+00
On time     9.158e-18   1.000e+00   2.391e-37  1.155e-192   0.000e+00
1-3d late  1.100e-200   2.391e-37   1.000e+00   1.437e-77  5.862e-175
4-7d late   0.000e+00  1.155e-192   1.437e-77   1.000e+00   2.544e-13
8d+ late    0.000e+00   0.000e+00  5.862e-175   2.544e-13   1.000e+00


## **Observation**

All **10 pairwise comparisons are statistically significant after Holm correction**, including **Early vs On time** (adjusted p = 9.16 × 10⁻¹⁸).

However, statistical significance should not be confused with practical importance. The **Early vs On-time difference is comparatively small** in absolute terms (mean review score: **4.29 vs 4.03**), while the differences become progressively larger as lateness increases. The strongest contrasts occur between the early/on-time groups and the materially late groups.

This distinction matters because the sample is large: with ~96,000 observations, even modest differences can produce extremely small p-values. The post-hoc results therefore establish **where differences exist**, while effect sizes and confidence intervals determine **how large and practically meaningful those differences are**.

## **Implication**

The pattern suggests that customer dissatisfaction is concentrated in **missed delivery commitments**, with substantially larger deterioration once orders move into the late bands. The relatively small Early vs On-time difference indicates that simply arriving earlier than promised may provide limited additional satisfaction compared with meeting the commitment.

The product implication should therefore be framed around **reliability against the promised delivery date**, rather than maximising how early orders arrive. This interpretation should be confirmed using the corresponding effect sizes and confidence intervals rather than p-values alone.

---

## **3.4 Effect Size — Cliff's Delta**

The Kruskal–Wallis test establishes that review-score distributions differ across delivery-lateness bands, while ε² summarizes the **overall magnitude** of that difference.

To quantify a specific pairwise contrast, we use **Cliff's delta** for **Early versus 8+ days late**. Cliff's delta is a non-parametric effect-size measure that represents the degree of stochastic dominance between two groups: it reflects how often an observation from one group has a higher outcome than an observation from the other, accounting for ties.

Unlike a p-value, Cliff's delta focuses on **magnitude rather than statistical significance**, making it useful for assessing whether the difference is practically meaningful.

In [9]:

# Cliff's Delta Effect Size

def cliffs_delta(x, y, cap=4000, seed=0):
    """
    Non-parametric effect size.

    Sub-sampled for tractability; the estimate is stable.
    """

    rng = np.random.default_rng(seed)

    x = rng.choice(
        x,
        min(cap, len(x)),
        replace=False
    )

    y = rng.choice(
        y,
        min(cap, len(y)),
        replace=False
    )

    gt = (
        x[:, None] > y[None, :]
    ).sum()

    lt = (
        x[:, None] < y[None, :]
    ).sum()

    return (gt - lt) / (
        len(x) * len(y)
    )


# Pairwise Comparisons

pairs = [
    ("Early", "8d+ late"),
    ("Early", "4-7d late"),
    ("Early", "1-3d late"),
    ("Early", "On time")
]


# Calculate and Display Cliff's Delta

print(
    "Cliff's delta  "
    "(|d|: 0.11 small | 0.28 medium | 0.43 large)\n"
)

for g1, g2 in pairs:

    d = cliffs_delta(
        a.review_score[
            a.lateness_band == g1
        ].values,
        a.review_score[
            a.lateness_band == g2
        ].values
    )

    mag = (
        "large"
        if abs(d) >= 0.43
        else "medium"
        if abs(d) >= 0.28
        else "small"
        if abs(d) >= 0.11
        else "negligible"
    )

    print(
        f"  {g1:<10} vs {g2:<10}  "
        f"d = {d:+.3f}  "
        f"({mag})"
    )

Cliff's delta  (|d|: 0.11 small | 0.28 medium | 0.43 large)

  Early      vs 8d+ late    d = +0.788  (large)
  Early      vs 4-7d late   d = +0.677  (large)
  Early      vs 1-3d late   d = +0.365  (medium)
  Early      vs On time     d = +0.127  (small)


## **Observation**

The effect sizes are substantial and increase sharply with lateness. Cliff's delta is **+0.788 for Early vs 8+ days late**, indicating a very large difference in the ranking of review outcomes. Specifically, the probability that an early-delivered order receives a higher review score than an 8+ day late order exceeds the probability of the reverse outcome by **78.8 percentage points**, with tied ratings contributing neither direction.

The pattern is graded rather than confined to extreme failures: **Early vs 4–7 days late has δ = +0.677 (large)**, while **Early vs 1–3 days late has δ = +0.365 (medium)**. Even modest lateness is therefore associated with a materially different distribution of customer ratings.

## **Implication**

The result confirms that the lateness–satisfaction relationship is not merely statistically significant; it is **large in practical magnitude**, with the difference becoming pronounced even before orders reach the most severe lateness category.

---

## **3.5 Poor-Review Proportion — χ² and Cramér's V**

Recasting the outcome as binary (`poor_review = 1` for ratings ≤ 2) provides a **business-legible measure of dissatisfaction**: the proportion of orders receiving a 1–2 star review.

A **chi-square test of independence** assesses whether poor-review rates differ systematically across delivery-lateness bands, while **Cramér's V** quantifies the strength of that association. This separates the question of **whether a relationship exists** from **how large the relationship is**.

In [10]:
# Chi-Square Test: Lateness vs Poor Review

ct = pd.crosstab(
    a.late_flag,
    a.poor_review
)

# 1. Chi-Square Test of Independence

chi2, p_chi, dof, _ = stats.chi2_contingency(ct)


# 2. Cramer's V Effect Size


V = np.sqrt(
    chi2 / (
        ct.values.sum() *
        (min(ct.shape) - 1)
    )
)


# 3. Poor-review Rates

late_rate = (
    100 * ct.loc[1, 1] /
    ct.loc[1].sum()
)

ontime_rate = (
    100 * ct.loc[0, 1] /
    ct.loc[0].sum()
)


print(
    f"chi2 = {chi2:,.1f}   "
    f"df = {dof}   "
    f"p = {p_chi:.3g}"
)

print(
    f"Cramer's V = {V:.3f}   "
    f"(0.1 small | 0.3 medium | 0.5 large)\n"
)

print(
    f"  poor-review rate, LATE    : "
    f"{late_rate:.1f}%   "
    f"(n = {ct.loc[1].sum():,})"
)

print(
    f"  poor-review rate, NOT LATE: "
    f"{ontime_rate:.1f}%   "
    f"(n = {ct.loc[0].sum():,})"
)

print(
    f"  relative risk             : "
    f"{late_rate / ontime_rate:.1f}x"
)

chi2 = 15,049.6   df = 1   p = 0
Cramer's V = 0.396   (0.1 small | 0.3 medium | 0.5 large)

  poor-review rate, LATE    : 62.4%   (n = 6,381)
  poor-review rate, NOT LATE: 9.3%   (n = 89,443)
  relative risk             : 6.7x


##**Observation**

The relationship is substantial on the business outcome of interest. **62.4% of late orders received a poor review (1–2 stars), compared with 9.3% of orders that were not late**. This corresponds to a **6.7× relative risk** of a poor review for late orders.

The chi-square test is highly significant (**χ² ≈ 15,050, p < 0.001**), and **Cramér's V ≈ 0.396**, indicating a **moderate-to-large association** between delivery lateness and poor-review outcomes.

The result is consistent with the earlier ordinal analysis: the relationship is not simply a statistically detectable difference in average review scores; **lateness is strongly associated with a much higher probability of a materially negative customer outcome**.



---

## **3.6 Bootstrap Confidence Interval on the Gap**

The relative risk is a **point estimate** describing how much more likely a late order is to receive a poor review than a non-late order. To quantify the uncertainty around this estimate, we use a **non-parametric bootstrap**, repeatedly resampling the observed orders with replacement and recalculating the relative risk.

This approach does not require a specific theoretical sampling distribution and provides a **95% bootstrap confidence interval** for the relative risk. The interval indicates the range of effect sizes that are consistent with the observed data.

The key question is therefore not only **"How large is the observed relative risk?"**, but also **"How precisely is it estimated?"**

In [11]:
# Bootstrap Confidence Intervals:
# Poor-Review Rate Difference and Relative Risk

rng = np.random.default_rng(42)

late = a.poor_review[
    a.late_flag == 1
].values

notlate = a.poor_review[
    a.late_flag == 0
].values

diffs = []
ratios = []


# Bootstrap Sampling

for _ in range(2000):

    l = rng.choice(
        late,
        len(late),
        replace=True
    ).mean()

    n_ = rng.choice(
        notlate,
        len(notlate),
        replace=True
    ).mean()

    # Difference in percentage points
    diffs.append(
        100 * (l - n_)
    )

    # Relative risk
    ratios.append(
        l / n_
    )



# Display 95% Confidence Intervals

print(
    f"Difference in poor-review rate : "
    f"{np.mean(diffs):.1f} pp   "
    f"95% CI "
    f"[{np.percentile(diffs, 2.5):.1f}, "
    f"{np.percentile(diffs, 97.5):.1f}]"
)

print(
    f"Relative risk                  : "
    f"{np.mean(ratios):.2f}x     "
    f"95% CI "
    f"[{np.percentile(ratios, 2.5):.2f}, "
    f"{np.percentile(ratios, 97.5):.2f}]"
)

Difference in poor-review rate : 53.1 pp   95% CI [52.0, 54.3]
Relative risk                  : 6.73x     95% CI [6.55, 6.92]


## **Observation**

The bootstrap results show that the large gap in poor-review rates is **estimated precisely**, not merely statistically significant. Late orders have a **53.1 percentage-point higher poor-review rate** than non-late orders, with a **95% bootstrap CI of [52.0, 54.3] pp**. The corresponding **relative risk is 6.73×**, with a **95% CI of [6.55, 6.92]**.

Both confidence intervals are narrow relative to the size of the observed effect, indicating that the estimated gap is stable under repeated resampling of the order-level data.

## **Implication**

The uncertainty around the association is small: the data consistently indicate a **substantially higher risk of poor reviews among late orders**. This strengthens the practical case for treating delivery reliability as an important customer-experience issue, beyond simply reporting a very small p-value.

## **Interpretation note**

The bootstrap quantifies the uncertainty of the observed association; it does **not establish causality**. A late order may be a marker of a broader fulfilment failure that also drives dissatisfaction.

---

## **3.7 Continuous Lateness — Spearman Correlation**

The lateness bands used above are useful for interpretation, but they are still a **categorisation of a continuous variable**. To check whether the observed relationship is not an artefact of those chosen bands, this analysis uses the underlying `days_late` measure directly.

**Spearman's rank correlation** tests whether review score changes systematically as delivery lateness increases, without requiring the relationship to be linear or either variable to be normally distributed.

The purpose is therefore a **robustness check**: if the relationship observed across lateness bands is genuine, a similar monotonic pattern should also appear when `days_late` is treated as a continuous variable.

In [12]:
# Spearman Rank Correlations
rho_pos, p1 = stats.spearmanr(
    a.days_late_pos,
    a.review_score
)

rho_sign, p2 = stats.spearmanr(
    a.lateness_days,
    a.review_score
)

# Display Results

print(
    f"days_late_pos (0 if early) vs review : "
    f"rho = {rho_pos:+.3f}  "
    f"p = {p1:.3g}"
)

print(
    f"signed lateness vs review            : "
    f"rho = {rho_sign:+.3f}  "
    f"p = {p2:.3g}"
)

days_late_pos (0 if early) vs review : rho = -0.314  p = 0
signed lateness vs review            : rho = -0.176  p = 0


## **Observation.**

Both continuous lateness measures show a statistically significant **negative monotonic association** with review score:

- `days_late_pos` (early deliveries set to 0): **Spearman ρ = −0.314**
- Signed `lateness_days`: **Spearman ρ = −0.176**

The stronger association for `days_late_pos` is itself informative. It suggests that the relationship is **asymmetric**: moving from early delivery toward lateness is associated with substantially lower ratings, while being increasingly early does not provide an equivalent improvement in ratings.

This is consistent with the earlier post-hoc result showing only a relatively small difference between **Early and On-time** orders, compared with the much larger deterioration observed once orders become late.

Both tests report **p < 0.001**; however, given the large sample size, the magnitude of the correlation is more informative than statistical significance alone.

---

## **3.8 Logistic Regression — Does the Association Survive Controls?**

The descriptive and non-parametric analyses establish a strong association between delivery lateness and customer dissatisfaction. The next question is whether that relationship remains after accounting for other observable characteristics of the order.

If lateness is primarily acting as a proxy for factors such as **longer delivery times, greater distance, higher order value, freight burden, or larger baskets**, its coefficient should weaken substantially once these factors are controlled.

We therefore estimate three nested logistic-regression models for `poor_review`:

- **M1 — Unadjusted:** lateness only.
- **M2 — Operational controls:** lateness + delivery time + distance + order value + freight + basket size.
- **M3 — Full specification:** M2 + product-category and destination-state fixed effects.

Comparing the lateness coefficient and odds ratio across the three models shows whether the observed relationship is **robust to observable confounding** or is largely explained by differences in order composition and geography.

**Interpretation rule.** A persistent and statistically precise lateness effect across specifications strengthens the evidence for an independent association. It does **not** establish causality, since unobserved confounding and reverse causality may remain.

In [13]:

# Logistic Regression Models

m = a.dropna(
    subset=[
        "log_dist",
        "log_price",
        "log_freight",
        "delivery_days",
        "n_items"
    ]
).copy()

print(
    f"modelling N = {len(m):,}\n"
)


# Model 1 — Unadjusted

M1 = smf.logit(
    "poor_review ~ late_flag",
    data=m
).fit(
    disp=0
)


# Model 2 — + Continuous Controls

M2 = smf.logit(
    "poor_review ~ late_flag + delivery_days + log_dist + log_price "
    "+ log_freight + n_items",
    data=m
).fit(
    disp=0
)


# Model 3 — + Category and Customer State

M3 = smf.logit(
    "poor_review ~ late_flag + delivery_days + log_dist + log_price "
    "+ log_freight + n_items + C(cat) + C(customer_state)",
    data=m
).fit(
    disp=0
)


# Display Model Results

print(
    f"{'model':<22}"
    f"{'OR':>8}"
    f"{'95% CI':>20}"
    f"{'p':>12}"
    f"{'pseudo R2':>12}"
)

for name, mod in [
    ("M1 unadjusted", M1),
    ("M2 + controls", M2),
    ("M3 + cat & state", M3)
]:

    b = mod.params["late_flag"]

    lo, hi = np.exp(
        mod.conf_int().loc["late_flag"]
    )

    print(
        f"{name:<22}"
        f"{np.exp(b):>8.2f}"
        f"{f'[{lo:.2f}, {hi:.2f}]':>20}"
        f"{mod.pvalues['late_flag']:>12.3g}"
        f"{mod.prsquared:>12.4f}"
    )

modelling N = 95,349

model                       OR              95% CI           p   pseudo R2
M1 unadjusted            16.25      [15.37, 17.18]           0      0.1316
M2 + controls             7.92        [7.31, 8.58]           0      0.1623
M3 + cat & state          7.80        [7.19, 8.46]           0      0.1658


## **Observation**

Delivery lateness remains a strong predictor of poor reviews after adjustment, but the magnitude of the association falls substantially once correlated fulfilment factors are introduced.

| Model | Specification | Odds Ratio | 95% CI |
|---|---|---:|---|
| **M1** | Lateness only | **16.25** | [15.37, 17.18] |
| **M2** | + delivery time, distance, order value, freight, basket size | **7.92** | [7.31, 8.58] |
| **M3** | + product category and destination-state fixed effects | **7.80** | [7.19, 8.46] |

The unadjusted model estimates **16.25× higher odds** of a poor review for late orders. After adding operational controls in M2, the estimate falls to **7.92×**, indicating that a substantial portion of the raw association overlaps with correlated fulfilment characteristics—particularly overall delivery time. The estimate then changes very little in M3, remaining at **7.80×** after controlling for product category and destination state.

## **Implication**

The attenuation from 16.25× to approximately **7.8×** is important: the raw association substantially overstates the relationship, but the association **does not disappear after adjustment**. Within the observable factors included in these models, late orders continue to have much higher odds of receiving a poor review.

The result is therefore **large, precisely estimated, and robust to the controls tested**. However, the odds ratio should be interpreted as an adjusted association, **not a causal effect**. Unobserved confounding and the possibility that lateness is a marker of broader fulfilment failures remain.

---

## **3.9 H1 Conclusion**

**Statistical significance.** The relationship is highly significant across the primary analyses (**p < 0.001**). Given the large sample size, however, significance alone is not the basis for the conclusion.

**Effect size.** The association is consistently substantial across complementary measures: **ε² = 0.106**, **Cliff's δ = 0.79** for Early vs 8+ days late, **Cramér's V = 0.396**, and an adjusted **odds ratio of 7.80** in the full logistic model.

**Practical significance.** **62.4% of late orders receive a 1–2 star review, compared with 9.3% of non-late orders**, corresponding to a **53.1 percentage-point gap** and a **6.7× relative risk**. Approximately **6,400 orders** fall into the three late bands, representing around **6.6% of delivered orders**. The affected population is therefore material but sufficiently contained to make targeted operational intervention plausible.

**Causal interpretation — what the data cannot establish.**

1. **Reverse causality remains possible.** A lost, damaged, or misrouted order may simultaneously be late and poorly reviewed. Lateness may therefore be a **marker of broader fulfilment failure**, rather than the direct mechanism producing dissatisfaction.
2. **The lateness definition is benchmark-relative.** Approximately 92% of orders arrive before the estimated date, so a "late" order has exceeded an already-buffered promise and may represent an unusually severe fulfilment event.
3. **Unobserved confounding remains possible.** Factors such as packaging quality, product condition, or seller communication are not captured and may contribute to both lateness and dissatisfaction.

### **Conclusion**

> **Delivery lateness is strongly associated with poor customer reviews. In the fully adjusted logistic model, late orders have approximately 7.8× the odds of receiving a 1–2 star review relative to non-late orders (OR = 7.80, 95% CI [7.19, 8.46], p < 0.001), controlling for delivery time, distance, order value, freight, basket size, product category, and destination state. The association is large and precisely estimated, but because the data are observational and lateness may be a marker of broader fulfilment failure, the estimate should not be interpreted as a causal effect.**

---# 4 · H2 — Distance and fulfilment burden**Business question.** Does geographic distance justify investment in distributed supply?Notebook 02 reported that the distance coefficient on review score **turns positive** oncedelivery time is controlled. That sign flip needs explaining rather than presenting.

In [14]:
print("Spearman correlations with seller-customer distance:\n")

for y, label in [
    ("delivery_days", "Delivery days"),
    ("freight_value", "Freight (R$)"),
    ("review_score", "Review score")
]:
    r, p = stats.spearmanr(m["max_distance_km"], m[y])
    print(f"  {label:<16} rho = {r:+.3f}   p = {p:.3g}")

Spearman correlations with seller-customer distance:

  Delivery days    rho = +0.540   p = 0
  Freight (R$)     rho = +0.589   p = 0
  Review score     rho = -0.067   p = 1.81e-96


**Expected: delivery ρ = +0.542, freight ρ = +0.584, review ρ = −0.064.**The asymmetry is the headline. Distance is **strongly** associated with time and cost, andonly **negligibly** with satisfaction (|ρ| < 0.1 is conventionally negligible). Distance isa cost-and-duration problem, not a satisfaction problem.

---
---

## **4.1 The Sign Flip — Is It Mediation or Multicollinearity?**

When delivery time is added to the distance model, the coefficient on `log_distance` changes sign. A natural concern is that **distance and delivery time are so strongly correlated that the regression coefficients become unstable**. This section checks that explanation directly using **Variance Inflation Factors (VIFs)**.

**Question.** Is the sign change in the distance coefficient a multicollinearity problem, or does it reflect the fact that part of distance's relationship with customer experience operates through delivery time?

**Method.** VIF measures how much the variance of an estimated coefficient is inflated because of linear dependence among predictors. Values above 5 are commonly treated as a warning sign, with values above 10 indicating serious concern.


In [15]:
# Variance Inflation Factor (VIF)


X = m[
    [
        "days_late_pos",
        "delivery_days",
        "log_dist",
        "log_freight",
        "log_price"
    ]
].astype(float)

X["const"] = 1.0

# Calculate VIF

print(
    "Variance inflation factors  "
    "(>5 concerning, >10 serious):\n"
)

for i, c in enumerate(X.columns):

    if c == "const":
        continue

    print(
        f"  {c:<16} "
        f"VIF = {variance_inflation_factor(X.values, i):.2f}"
    )

Variance inflation factors  (>5 concerning, >10 serious):

  days_late_pos    VIF = 2.18
  delivery_days    VIF = 2.60
  log_dist         VIF = 1.61
  log_freight      VIF = 1.68
  log_price        VIF = 1.30


## **Observation**

All continuous predictors have **VIF values between 1.30 and 2.60**:

| Predictor | VIF |
|---|---:|
| `days_late_pos` | 2.18 |
| `delivery_days` | 2.60 |
| `log_dist` | 1.61 |
| `log_freight` | 1.68 |
| `log_price` | 1.30 |

These values are comfortably below conventional concern thresholds, indicating **no material multicollinearity among the continuous predictors** in this specification.

The sign change in the `log_dist` coefficient is therefore unlikely to be a numerical artefact caused by unstable collinearity. Instead, it is consistent with a **mediation/suppression pattern**: distance is strongly related to delivery time, and once delivery time is held constant, the remaining association between distance and review outcome changes direction.

## **Interpretation caution**

VIF rules out substantial linear multicollinearity as the main explanation, but it does **not prove mediation**. Establishing a formal mediation pathway would require an explicit mediation analysis and stronger assumptions. The defensible conclusion here is that the sign flip **is not explained by problematic multicollinearity and warrants interpretation through the relationship between distance and delivery time**.

---

## **4.2 Nested Regression — What Happens When Delivery Time Is Held Constant?**

The previous analysis shows that seller–customer distance is associated with delivery time, freight, and review outcomes. This raises an important question: **is distance itself associated with customer satisfaction, or does its relationship with reviews operate primarily through the time customers spend waiting?**

To investigate this, three nested OLS models are compared:

- **Model A — Distance only:** `review_score ~ log_distance`
- **Model B — + Delivery time:** adds `delivery_days`
- **Model C — + Lateness & controls:** further adds lateness and relevant order-level controls

The key quantity is the coefficient on `log_distance`. Comparing its **sign, magnitude, and explanatory power** across models shows how the distance–review relationship changes as fulfilment variables are introduced.

In [16]:
# OLS Models: Effect of Distance on Review Score

A = smf.ols(
    "review_score ~ log_dist",
    data=m
).fit()

B = smf.ols(
    "review_score ~ log_dist + delivery_days",
    data=m
).fit()

C = smf.ols(
    "review_score ~ log_dist + delivery_days + days_late_pos + log_price "
    "+ log_freight + C(cat)",
    data=m
).fit()



# Display Results

print(
    f"{'model':<26}"
    f"{'log_dist beta':>16}"
    f"{'p':>12}"
    f"{'R2':>10}"
)

for name, mod in [
    ("A: distance only", A),
    ("B: + delivery_days", B),
    ("C: + lateness & controls", C)
]:
    print(
        f"{name:<26}"
        f"{mod.params['log_dist']:>+16.4f}"
        f"{mod.pvalues['log_dist']:>12.3g}"
        f"{mod.rsquared:>10.4f}"
    )

model                        log_dist beta           p        R2
A: distance only                   -0.0625    9.38e-94    0.0044
B: + delivery_days                 +0.0756   4.03e-129    0.1171
C: + lateness & controls           +0.1001   4.23e-165    0.1237


## **Observation**

The distance coefficient changes direction once delivery time is introduced:

| Model | Specification | `log_distance` β | p-value | R² |
|---|---|---:|---:|---:|
| **A** | Distance only | **−0.0625** | 9.38 × 10⁻⁹⁴ | 0.0044 |
| **B** | + Delivery time | **+0.0756** | 4.03 × 10⁻¹²⁹ | 0.1171 |
| **C** | + Lateness & controls | **+0.1001** | 4.23 × 10⁻¹⁶⁵ | 0.1237 |

In **Model A**, distance has a small negative association with review score. However, once delivery time is controlled for in **Model B**, the coefficient reverses sign and becomes positive. Adding lateness and the remaining controls in **Model C** strengthens the positive coefficient further.

The important result is therefore not the sign by itself, but the **change in the relationship after accounting for fulfilment time**. This is consistent with a mediation or suppression pattern in which distance is closely linked to how long an order takes to arrive, while the observed satisfaction penalty is more closely related to the waiting experience than to geographic distance itself.

At the same time, the distance-only model explains just **0.44% of the variation in review score**, so distance is a weak predictor of satisfaction in absolute terms despite its highly significant coefficient.

## **Interpretation**

Conditional on comparable delivery time and other observed factors, more distant orders have a slightly higher predicted review score in these models. One plausible explanation is that longer-distance orders may receive longer promised delivery windows, which can moderate customer expectations. This is an **observational interpretation**, not evidence that distance improves satisfaction.

## **Caveat**
he coefficient reversal is consistent with a mediation-type relationship, but these regressions do **not constitute a formal mediation analysis**. The interpretation assumes delivery time lies on the pathway from distance to customer experience rather than acting as a confounder; the temporal ordering supports this interpretation but does not establish it causally.

---

## **5. H4 — Cross-State vs Within-State Fulfilment**

**Question.** Does the geographic supply–demand mismatch identified in Chapter 2 translate into a measurable fulfilment burden?

Instead of inferring this relationship from state-level supply and demand shares, this analysis compares **within-state and cross-state orders directly** on seller–customer distance, delivery time, freight, and customer satisfaction.

**Grain.** One row per order, with `is_interstate` indicating whether the seller and customer are located in different states.

**Test.** Because the fulfilment metrics are not assumed to be normally distributed, the two groups are compared using the **Mann–Whitney U test**. **Rank-biserial correlation** is reported as the effect size.

In [17]:

# Same-State vs Cross-State Orders

g = m.groupby("is_interstate").agg(
    orders=("order_id", "size"),
    distance_km=("max_distance_km", "mean"),
    delivery_days=("delivery_days", "mean"),
    freight=("freight_value", "mean"),
    review_score=("review_score", "mean"),
    pct_poor=("poor_review", lambda s: 100 * s.mean()),
).round(2)

g.index = [
    "Same state",
    "Cross state"
]

print(
    g.to_string(),
    "\n"
)



# Mann-Whitney U Tests + Rank-Biserial Effect Size

for col in [
    "delivery_days",
    "freight_value",
    "max_distance_km"
]:

    u = stats.mannwhitneyu(
        m[col][m.is_interstate == 0],
        m[col][m.is_interstate == 1]
    )

    # Sample sizes
    n1 = (m.is_interstate == 0).sum()
    n2 = (m.is_interstate == 1).sum()

    # Rank-biserial correlation as effect size
    rb = 1 - (
        2 * u.statistic
    ) / (n1 * n2)

    print(
        f"  {col:<18} "
        f"Mann-Whitney p = {u.pvalue:.3g}   "
        f"rank-biserial r = {rb:+.3f}"
    )

             orders  distance_km  delivery_days  freight  review_score  pct_poor
Same state    34417     155.1900         7.8700  15.4300        4.2600   10.4700
Cross state   60932     854.0400        15.0300  26.9000        4.0900   14.1400 

  delivery_days      Mann-Whitney p = 0   rank-biserial r = +0.593
  freight_value      Mann-Whitney p = 0   rank-biserial r = +0.669
  max_distance_km    Mann-Whitney p = 0   rank-biserial r = +0.912


## **Observation**

Cross-state fulfilment carries a substantially larger operational burden across all three fulfilment measures:

| Metric | Same state | Cross state | Difference |
|---|---:|---:|---:|
| Orders | 34,417 | 60,932 | — |
| Distance | 155 km | 854 km | **+451%** |
| Delivery time | 7.87 days | 15.03 days | **+91%** |
| Freight | R$15.43 | R$26.90 | **+74%** |
| Review score | 4.26 | 4.09 | **−0.17 pts** |
| Poor-review rate | 10.47% | 14.14% | **+3.67 pp** |

The differences in **distance, delivery time, and freight are all highly statistically significant (p < 0.001)**, with large rank-based effect sizes:

- **Distance:** rank-biserial r = **+0.912**
- **Freight:** rank-biserial r = **+0.669**
- **Delivery time:** rank-biserial r = **+0.593**

The review-score difference is much smaller in magnitude despite the large operational differences, reinforcing the earlier finding that **geographic distance primarily manifests as a fulfilment-cost and duration problem rather than a large direct satisfaction penalty**.

Approximately **64% of orders cross a state boundary**, making cross-state fulfilment the dominant operating pattern rather than a marginal exception.

## **Implication**

This provides the quantitative link that the descriptive geographic analysis could only suggest: **geographic dispersion is associated with materially greater fulfilment distance, delivery time, and freight burden**. The effect is particularly pronounced for distance itself, while the customer-experience difference is comparatively modest.

## **Limitation**

The comparison is observational. Cross-state orders differ from within-state orders in other respects—such as product mix, seller characteristics, order value, and destination—which may contribute to the observed differences. The results therefore establish **association, not causation**.

# 6 · **H3 — First-Order Satisfaction and Repeat Purchase**

**The most consequential hypothesis in the project.**  
If poor first-order satisfaction predicts whether customers return, then improving the fulfilment experience could plausibly contribute to higher retention. If no meaningful relationship exists, the retention problem must be investigated separately from delivery performance.

**Prerequisite.** The raw **3.12% ever-repeated** figure cannot be used as the retention outcome because the dataset is **right-censored**: customers acquired near the end of the observation period have substantially less time to place another order than customers acquired earlier.

The analysis therefore uses **fixed, censoring-aware repeat-purchase windows**—particularly the **90-day repeat indicator**—and includes only customers with sufficient observation time to have been observed for the full window.

**Unit of analysis.** One row per `customer_unique_id`, using the customer's **first observed order** as the baseline for measuring subsequent repeat behaviour.

**Primary question.** Does **first-order review satisfaction** predict whether a customer places a second order within 90 days?


## **6.1 Censoring-Corrected Repeat Rates**

The raw **3.12% ever-repeat rate** is not a valid retention measure because customers enter the dataset at different times and therefore have unequal opportunities to return. This section removes that exposure bias by measuring repeat purchase within fixed follow-up windows after the customer's first order.

For each window, only customers with enough observable follow-up time are considered **eligible**. This provides comparable 30-, 60-, 90-, and 180-day repeat rates and allows us to identify when repeat behaviour actually occurs.

In [18]:
# Repeat Purchase / Retention Rates

first = df[df.is_first_order].copy()

rows = []

for w, col in [
    (30, "repeat_within_30d"),
    (60, "repeat_within_60d"),
    (90, "repeat_within_90d"),
    (180, "repeat_within_180d")
]:
    # NULL = insufficient exposure
    e = first[first[col].notna()]

    rep = e[col].sum()
    rate = 100 * rep / len(e)

    se = (
        np.sqrt(
            (rate / 100)
            * (1 - rate / 100)
            / len(e)
        )
        * 100
    )

    rows.append([
        f"{w}d",
        len(e),
        int(rep),
        round(rate, 2),
        round(rate - 1.96 * se, 2),
        round(rate + 1.96 * se, 2)
    ])


# Retention Summary Table

out = pd.DataFrame(
    rows,
    columns=[
        "window",
        "eligible",
        "repeats",
        "rate_pct",
        "ci_low",
        "ci_high"
    ]
)

print(
    out.to_string(index=False)
)

print(
    f"\nuncensored 'ever repeated' : "
    f"{100 * first.ever_repeated.mean():.2f}%  "
    f"<- do not report as retention"
)

window  eligible  repeats  rate_pct  ci_low  ci_high
   30d     95388     1506    1.5800  1.5000   1.6600
   60d     94065     1826    1.9400  1.8500   2.0300
   90d     86445     2029    2.3500  2.2500   2.4500
  180d     68743     2455    3.5700  3.4300   3.7100

uncensored 'ever repeated' : 3.10%  <- do not report as retention


## **Observation**

Repeat purchase remains low even after correcting for unequal observation time, but it increases predictably with the length of the follow-up window:

| Window | Eligible customers | Repeat customers | Repeat rate | 95% CI |
|---|---:|---:|---:|---:|
| 30 days | 95,388 | 1,506 | **1.58%** | [1.50%, 1.66%] |
| 60 days | 94,065 | 1,826 | **1.94%** | [1.85%, 2.03%] |
| 90 days | 86,445 | 2,029 | **2.35%** | [2.25%, 2.45%] |
| 180 days | 68,743 | 2,455 | **3.57%** | [3.43%, 3.71%] |

The key pattern is the **shape of the curve**: repeat purchasing increases rapidly during the first 90 days and then grows more slowly. By 90 days, **2.35% of eligible first-time customers have returned**, compared with 3.57% within 180 days.

This also explains why the censoring correction does not produce a higher rate than the raw 3.12% ever-repeat figure. The fixed-window rates answer a different question: **what proportion of customers return within a defined period when everyone has the same opportunity to do so?**

## **Implication**

Repeat behaviour is both **rare and front-loaded**. Customers who return tend to do so relatively early, while additional repeat activity after 90 days is limited. For the remainder of the project, **90-day repeat purchase is adopted as the headline retention outcome**, since it captures a substantial share of observed repeat behaviour while retaining a large eligible population.

## **6.2 Does First-Order Review Score Predict Repeat Purchase?**

The central retention question is whether a customer's **first-order experience is associated with their likelihood of returning**. Before fitting a regression model, we first examine the 90-day repeat-purchase rate across first-order review scores.

**Outcome.** `repeat_within_90d` indicates whether the customer placed another order within 90 days of their first purchase.

The descriptive comparison provides an initial check for a systematic relationship; the subsequent logistic regression tests whether any observed difference remains after accounting for other customer and order characteristics.

In [19]:

# 90-Day Repeat Purchase by First-Order Review Score

fo = df[
    df.is_first_order
    & df.repeat_within_90d.notna()
].copy()

fo["rep90"] = (
    fo.repeat_within_90d.astype(int)
)

fo["log_dist"] = (
    np.log1p(fo.max_distance_km)
)

fo["log_price"] = (
    np.log1p(fo.item_value)
)

fo = fo.dropna(
    subset=[
        "review_score",
        "log_dist",
        "log_price",
        "delivery_days",
        "days_late_pos"
    ]
)

print(
    f"N = {len(fo):,}   "
    f"positives = {fo.rep90.sum():,} "
    f"({100 * fo.rep90.mean():.2f}%)\n"
)

# Unadjusted 90-Day Repeat Rate by Review Score

print(
    "Unadjusted 90-day repeat rate "
    "by first-order review score:\n"
)

gg = (
    fo.groupby("review_score")
    .rep90
    .agg(["mean", "count"])
)

gg["rate"] = (
    100 * gg["mean"]
)

gg["se"] = (
    np.sqrt(
        gg["mean"]
        * (1 - gg["mean"])
        / gg["count"]
    )
    * 100
)

for s, r in gg.iterrows():
    print(
        f"  {int(s)} star : "
        f"n = {int(r['count']):>6,}   "
        f"rate = {r['rate']:.2f}%   "
        f"95% CI "
        f"[{r['rate'] - 1.96 * r['se']:.2f}, "
        f"{r['rate'] + 1.96 * r['se']:.2f}]"
    )

N = 83,462   positives = 1,945 (2.33%)

Unadjusted 90-day repeat rate by first-order review score:

  1 star : n =  8,424   rate = 2.64%   95% CI [2.29, 2.98]
  2 star : n =  2,597   rate = 2.46%   95% CI [1.87, 3.06]
  3 star : n =  7,030   rate = 2.36%   95% CI [2.01, 2.72]
  4 star : n = 16,591   rate = 2.07%   95% CI [1.85, 2.28]
  5 star : n = 48,820   rate = 2.36%   95% CI [2.22, 2.49]


## **Observation**

The 90-day repeat rate shows **no clear monotonic relationship with first-order review score**. Repeat rates remain clustered tightly around 2–3% across all five rating groups:

| First-order review | Eligible customers | 90-day repeat | 95% CI |
|---|---:|---:|---:|
| 1 star | 8,424 | **2.64%** | [2.29%, 2.98%] |
| 2 stars | 2,597 | **2.46%** | [1.87%, 3.06%] |
| 3 stars | 7,030 | **2.36%** | [2.01%, 2.72%] |
| 4 stars | 16,591 | **2.07%** | [1.85%, 2.28%] |
| 5 stars | 48,820 | **2.36%** | [2.22%, 2.49%] |

There is no consistent increase or decrease in repeat purchase as review score improves. The highest observed repeat rate is actually among **1-star customers (2.64%)**, while **5-star customers return at 2.36%**. The differences are small relative to the overall rarity of repeat purchase, and the confidence intervals overlap substantially across the groups.

## **Implication**

Descriptively, **first-order satisfaction does not appear to be a strong driver of 90-day repeat purchase**. This contrasts sharply with the strong relationship between delivery lateness and review outcomes found earlier: an order can be associated with a much poorer review without that dissatisfaction translating into an obvious difference in near-term repeat behaviour.

This is only a descriptive comparison; the formal conclusion comes from the logistic regression, which tests the association while controlling for other observable characteristics.

---

## **6.3 Multivariable Logistic Regression — Does First-Order Satisfaction Predict Repeat Purchase?**

The unadjusted comparison shows little systematic difference in 90-day repeat purchase across first-order review scores. This regression asks whether that relationship changes after accounting for other observable characteristics of the first order.

**Outcome.** `repeat_within_90d = 1` if the customer places another order within 90 days.

**Predictors.** First-order `review_score`, positive lateness, delivery time, seller–customer distance, and first-order price.

The model reports **odds ratios, 95% confidence intervals, and p-values**. The primary focus is the effect of first-order review score: whether satisfaction remains associated with near-term repeat purchase after accounting for observable order characteristics.

In [20]:

# Logistic Regression: 90-Day Repeat Purchase

lg = smf.logit(
    "rep90 ~ review_score + days_late_pos + delivery_days "
    "+ log_dist + log_price",
    data=fo
).fit(disp=0)

# Convert coefficient confidence intervals to Odds Ratios
ci = np.exp(
    lg.conf_int()
)


# Display Model Results

print(
    f"{'variable':<16}"
    f"{'OR':>9}"
    f"{'95% CI':>22}"
    f"{'p':>12}"
)

for v in [
    "review_score",
    "days_late_pos",
    "delivery_days",
    "log_dist",
    "log_price"
]:
    print(
        f"{v:<16}"
        f"{np.exp(lg.params[v]):>9.4f}"
        f"{f'[{ci.loc[v, 0]:.4f}, {ci.loc[v, 1]:.4f}]':>22}"
        f"{lg.pvalues[v]:>12.4g}"
    )

print(
    f"\npseudo R2 = {lg.prsquared:.5f}"
)

variable               OR                95% CI           p
review_score       0.9560      [0.9219, 0.9914]     0.01538
days_late_pos      1.0001      [0.9833, 1.0172]      0.9926
delivery_days      0.9926      [0.9847, 1.0006]     0.06994
log_dist           0.9685      [0.9326, 1.0058]     0.09654
log_price          0.9301      [0.8848, 0.9776]    0.004388

pseudo R2 = 0.00154


## **Observation**

First-order review score shows a **small but statistically significant negative association** with 90-day repeat purchase after adjustment:

- `review_score`: **OR = 0.956**, 95% CI **[0.922, 0.991]**, **p = 0.015**
- `days_late_pos`: **OR = 1.000**, 95% CI **[0.983, 1.017]**, p = 0.993
- `delivery_days`: **OR = 0.993**, 95% CI **[0.985, 1.001]**, p = 0.070
- `log_dist`: **OR = 0.969**, 95% CI **[0.933, 1.006]**, p = 0.097
- `log_price`: **OR = 0.930**, 95% CI **[0.885, 0.978]**, **p = 0.004**

For `review_score`, an odds ratio of **0.956** means that each additional review star is associated with approximately a **4.4% decrease in the odds of a 90-day repeat purchase**, holding the other variables in the model constant. Although statistically significant at the 5% level, the effect is **small in practical terms**. Moving from 1 to 5 stars corresponds to approximately 0.956⁴ ≈ **0.83**, or about **17% lower estimated odds**, under the model's linear-on-the-log-odds specification.

The model's **McFadden pseudo-R² is only 0.00154**, indicating that the included variables explain very little of the variation in repeat-purchase behaviour.

The other predictors provide limited evidence of an association at the 5% level. Delivery time and distance are directionally negative but do not reach conventional significance, while **higher first-order price is significantly associated with lower repeat-purchase odds** (OR = 0.930, p = 0.004).

## **Implication**

First-order review score is **statistically associated with repeat purchase in this specification, but the effect is small and the model has very limited explanatory power**. The result therefore does not support a simple conclusion that improving customer satisfaction will materially solve the retention problem. The stronger signal may lie in other customer or marketplace characteristics not captured by this model.

## **Important interpretation note**

Statistical significance should not be treated as evidence of a practically important effect, particularly with a large sample. The negative review-score coefficient is also counterintuitive and should be treated as an empirical result requiring further investigation rather than interpreted as evidence that better customer experiences reduce retention.

## **6.3 H3 Conclusion — First-Order Satisfaction Does Not Explain the Retention Gap**

### **Statistical result**

First-order review score shows a **small but statistically significant negative association** with 90-day repeat purchase in the current logistic model (**OR = 0.956, 95% CI [0.922, 0.991], p = 0.015**). However, the effect is modest: each additional review star is associated with approximately a **4.4% decrease in the odds of returning**, holding the other variables in the model constant. The model itself has very limited explanatory power (**McFadden pseudo-R² = 0.00154**).

**This changes but does not overturn—the project's original narrative.** The descriptive and fulfilment analyses establish a strong relationship between delivery performance and customer dissatisfaction. However, the retention model provides **no evidence that better first-order reviews translate into materially higher repeat purchase**. In fact, the estimated association is slightly negative and counterintuitive, suggesting that first-order satisfaction is not a meaningful standalone explanation for why customers return.

The project therefore separates two distinct marketplace problems:

| | Problem | Evidence | Implication |
|---|---|---|---|
| **1** | **Fulfilment / CX** | Late orders have substantially higher odds of poor reviews; adjusted OR ≈ **7.8** | Large association; operationally important |
| **2** | **Retention** | 90-day repeat rate is only **~2.3%**, while first-order review has only a small association with repeat purchase | Delivery improvement alone is unlikely to explain or solve the retention gap |

### **Implication**

Improving delivery remains justified as a **customer-experience and fulfilment initiative**, given the large association with poor reviews. But the evidence does **not** support treating delivery-related dissatisfaction as the primary explanation for low repeat purchasing. Retention should therefore be investigated as a separate product problem.

### **What might explain the weak repeat behaviour?**

The current analysis does not establish the underlying causes, but plausible areas for further investigation include:

- **Purchase occasion:** some product categories may naturally generate one-off purchases rather than recurring demand.
- **Marketplace relationship:** customers may transact through Olist without developing strong attachment to the marketplace itself.
- **Re-engagement:** the dataset does not directly capture lifecycle marketing, recommendations, reminders, or other mechanisms that encourage a second purchase.
- **Order value:** larger first orders are negatively associated with 90-day repeat purchase in the current model (**OR = 0.930 for log price, p = 0.004**), which is consistent with but does not prove a more considered, one-off purchasing pattern.

### **Reporting principle**

The unexpected direction of the review-score coefficient should be reported rather than hidden. It is a small association, the model explains very little of repeat-purchase behaviour, and the result should not be interpreted as evidence that better customer experiences *reduce* retention.

**Conclusion.**

> **First-order satisfaction is not a strong practical explanation for Olist's low repeat-purchase rate. Although review score is statistically associated with 90-day repeat purchase in the current model, the effect is small and the model has very low explanatory power. The evidence therefore supports treating fulfilment/CX and retention as distinct product problems rather than assuming that improving delivery will materially increase repeat purchasing.**

# **7. Seller Concentration**

Seller concentration is treated as a **descriptive marketplace-structure measure**, not a hypothesis test. There is no meaningful null hypothesis to reject about how unequal the seller-value distribution should be.

The analysis therefore quantifies concentration using **seller-share measures, Gini, and HHI**, with particular attention to how much marketplace item value is controlled by the highest-value sellers.

In [21]:

# Seller Concentration Analysis


sellers = q("""
    SELECT
        seller_id,
        SUM(price) AS item_value
    FROM `olist.v_fact_order_items`
    GROUP BY seller_id
""")

# Sort sellers by item value in descending order
rev = np.sort(
    sellers.item_value.values
)[::-1]

# Calculate each seller's share of total item value
share = rev / rev.sum()


# Gini Coefficient

def gini(x):
    x = np.sort(x)

    n = len(x)
    idx = np.arange(1, n + 1)

    return (
        (2 * np.sum(idx * x) / (n * np.sum(x)))
        - (n + 1) / n
    )



# HHI — Herfindahl-Hirschman Index

hhi = (
    (share ** 2).sum()
    * 10_000
)

n = len(rev)

# Normalised HHI removes the effect of the number of sellers
hhi_norm = (
    (
        (share ** 2).sum()
        - 1 / n
    )
    / (
        1 - 1 / n
    )
    * 10_000
)



# Overall Concentration Metrics

print(
    f"sellers = {n:,}"
)

print(
    f"Gini             = {gini(rev):.3f}"
)

print(
    f"HHI              = {hhi:,.0f}   "
    f"(normalised {hhi_norm:,.0f})\n"
)


# Cumulative Seller Concentration

cum = np.cumsum(share)

for p in [1, 5, 10, 20, 50]:
    k = max(
        1,
        int(round(n * p / 100))
    )

    print(
        f"  top {p:>2}% "
        f"({k:>4} sellers) : "
        f"{100 * cum[k - 1]:.1f}% "
        f"of item value"
    )

sellers = 3,095
Gini             = 0.792
HHI              = 36   (normalised 32)

  top  1% (  31 sellers) : 26.1% of item value
  top  5% ( 155 sellers) : 53.3% of item value
  top 10% ( 310 sellers) : 67.6% of item value
  top 20% ( 619 sellers) : 82.7% of item value
  top 50% (1548 sellers) : 96.8% of item value


**Observation.** Seller value is highly unevenly distributed across the **3,095 sellers**:

| Metric | Result |
|---|---:|
| Gini coefficient | **0.792** |
| HHI | **36** |
| Normalised HHI | **32** |
| Top 1% (31 sellers) | **26.1%** of item value |
| Top 5% (155 sellers) | **53.3%** |
| Top 10% (310 sellers) | **67.6%** |
| Top 20% (619 sellers) | **82.7%** |
| Top 50% (1,548 sellers) | **96.8%** |

The **Gini coefficient of 0.792** indicates substantial inequality in the distribution of seller-level item value: marketplace activity is heavily concentrated among a relatively small subset of sellers.

The concentration is particularly pronounced at the top: **5% of sellers account for 53.3% of item value, 10% account for 67.6%, and 20% account for 82.7%**, while the bottom half contributes only **3.2%**.

**Metric caveat.** HHI is much less informative in this setting. With 3,095 sellers, individual seller shares are small enough that the sum of squared shares remains low (**HHI = 36**), which could misleadingly suggest an unconcentrated marketplace under conventional HHI thresholds. Here, HHI is better treated as a reference measure; **Gini and seller-share concentration ratios more clearly capture the inequality of the seller-value distribution**.

**Implication.** Concentration creates both an opportunity and a risk. A targeted programme reaching roughly **600 high-value sellers could cover ~83% of item value**, making focused intervention potentially efficient. At the same time, substantial dependence on a relatively small seller base creates **marketplace concentration risk**, since disruption or quality deterioration among high-value sellers could have a disproportionate effect on overall marketplace activity.

# **9. Summary**

### **What the evidence supports**

**Delivery lateness is strongly associated with poor customer reviews.** Late orders have substantially higher odds of receiving a 1–2 star review, with an adjusted odds ratio of approximately **7.8** in the full logistic model. The relationship remains large after controlling for delivery time, distance, order value, freight, basket size, product category, and destination state, and is supported by multiple complementary effect-size measures.

**Distance is primarily a fulfilment-burden variable.** Greater seller–customer distance is strongly associated with longer delivery times and higher freight costs, while its direct association with review score is comparatively small. Cross-state orders reinforce this pattern: they travel substantially farther, take longer to arrive, and incur higher freight costs than within-state orders.

**Seller supply is highly concentrated.** The top 20% of sellers account for approximately **82.7% of item value**, while the Gini coefficient is **0.792**. This creates both an opportunity for targeted seller interventions and potential concentration risk.

### **What the evidence does not support**

**First-order satisfaction does not provide a strong practical explanation for repeat purchase.** The current 90-day repeat-purchase model finds a small statistically significant association between first-order review score and repeat purchase (**OR = 0.956, 95% CI [0.922, 0.991], p = 0.015]**), but the effect is modest and the model explains very little of the variation in repeat behaviour (**McFadden pseudo-R² ≈ 0.0015**). The direction is also counterintuitive.

The evidence therefore does **not** support the simple hypothesis that poor first-order satisfaction is the main reason customers fail to return.

### **What follows for the business**

**Fulfilment should still be improved—but not as a presumed retention lever.** Delivery lateness is strongly associated with customer dissatisfaction and represents a clear operational/CX problem. The relatively contained population of late orders also provides a well-defined target for proactive intervention.

**Retention requires a separate product investigation.** The weak explanatory power of the current repeat-purchase model suggests that improving delivery alone is unlikely to explain a substantial change in retention. Factors such as purchase category, customer lifecycle, marketplace engagement, and re-engagement mechanisms warrant separate investigation.

### **What cannot be claimed**

These findings are **associations, not causal estimates**.

The Olist dataset is observational and contains no randomized treatment assignment. In particular:

- A failed fulfilment event may be both **late and poorly reviewed**, so lateness may be a marker of broader service failure rather than its direct cause.
- Olist's delivery estimates are heavily padded, meaning "late" represents an order that exceeded an already-generous promised window.
- Important factors such as product condition, packaging, seller communication, and other unobserved customer-experience drivers are not fully captured.

The conclusion is therefore:

> **Olist has a clear fulfilment/CX problem and a separate retention problem. Delivery lateness is strongly and robustly associated with dissatisfaction, but the available evidence does not establish that improving fulfilment will materially increase repeat purchasing.**